# Logistic Regression — Airline Customer Satisfaction Prediction

This project builds a Logistic Regression classification model to predict whether an airline passenger is **Satisfied** or **Neutral/Dissatisfied** based on survey data.

## Step 1: Load Dataset and Inspect Target Variable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load dataset
df = pd.read_csv('airline_satisfaction.csv')
print('Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nTarget variable (satisfaction) distribution:')
print(df['satisfaction'].value_counts())
print('\nMissing values:')
print(df.isnull().sum())

## Step 2: Encode Categorical Variables and Handle Missing Values

In [ ]:
# Drop rows with missing values
df = df.dropna()

# Encode target variable: satisfied=1, neutral or dissatisfied=0
df['satisfaction_encoded'] = (df['satisfaction'] == 'satisfied').astype(int)
print('Target encoding:')
print(df['satisfaction_encoded'].value_counts())

# Encode categorical predictors
le = LabelEncoder()
df['Customer Type_enc'] = le.fit_transform(df['Customer Type'])
df['Type of Travel_enc'] = le.fit_transform(df['Type of Travel'])
df['Class_enc'] = le.fit_transform(df['Class'])

print('\nEncoding complete.')
print(df[['Customer Type', 'Customer Type_enc']].drop_duplicates())

## Step 3: Feature Selection and Train/Test Split

In [ ]:
# Select features for the model
feature_cols = [
    'Age', 'Flight Distance', 'Customer Type_enc', 'Type of Travel_enc', 'Class_enc',
    'Seat comfort', 'Inflight wifi service', 'Inflight entertainment',
    'Online support', 'Ease of Online booking', 'On-board service',
    'Leg room service', 'Baggage handling', 'Checkin service',
    'Cleanliness', 'Online boarding',
    'Departure Delay in Minutes', 'Arrival Delay in Minutes'
]

X = df[feature_cols]
y = df['satisfaction_encoded']

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training set size:', X_train.shape)
print('Testing set size: ', X_test.shape)

## Step 4: Scale Features and Build Logistic Regression Model

In [ ]:
# Scale features for better model convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Build Logistic Regression model (binomial — predicts Satisfied vs Not)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print('Model trained successfully!')
print('Classes:', model.classes_)

## Step 5: Evaluate Model — Confusion Matrix, Precision, Recall, Accuracy

In [ ]:
# Predict on test set
y_pred = model.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Satisfied', 'Satisfied'],
            yticklabels=['Not Satisfied', 'Satisfied'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Performance metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

print(f'\nAccuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print('\nFull Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Satisfied', 'Satisfied']))

## Step 6: Interpret Model Coefficients — Key Drivers of Satisfaction

In [ ]:
# Extract and sort coefficients
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print('Feature Coefficients (positive = increases satisfaction probability):')
print(coef_df.to_string(index=False))

# Plot top features
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linestyle='--')
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression Coefficients — Drivers of Passenger Satisfaction')
plt.tight_layout()
plt.show()

## Step 7: Business Recommendations and Limitations

### Key Findings

**Top drivers of satisfaction (positive coefficients):**
- **Online boarding** — passengers who use online boarding report higher satisfaction
- **Inflight entertainment** — strong positive impact on satisfaction
- **Seat comfort** — key differentiator for satisfied passengers
- **Business class travel** — business travelers tend to be more satisfied

**Factors reducing satisfaction (negative coefficients):**
- **Departure/Arrival delays** — delays significantly reduce satisfaction probability
- **Personal travel** — personal travelers tend to be less satisfied than business travelers

### Precision vs Recall Trade-off
- **Precision**: Of all passengers predicted as Satisfied, how many actually were?
- **Recall**: Of all actually Satisfied passengers, how many did the model correctly identify?
- High recall is important here — we want to catch as many truly satisfied customers as possible to understand retention drivers

### Business Recommendations
1. **Improve online boarding experience** — strongest predictor of satisfaction
2. **Invest in inflight entertainment** — high ROI for satisfaction improvement
3. **Reduce delays** — departure/arrival delays are the top driver of dissatisfaction
4. **Upgrade seat comfort in Economy class** — personal travelers in Eco class are least satisfied
5. **Target business travelers** for loyalty programs — they show highest satisfaction rates

### Limitations and Next Steps
- Model assumes linear relationship between features and log-odds of satisfaction
- Could improve with Random Forest or XGBoost for non-linear patterns
- Next step: Deploy model to flag at-risk passengers before flights for proactive intervention